### A. Using the mathematical representations developed in Task 2, write a program in either Python or R to solve the optimization problem computationally.

The goal is to determine how many tons Amazon should ship on each available route so that all fulfillment center demand is met at the lowest possible total cost.

### A1. Solver Solution

Before building the model, I organized the provided Excel data into separate tables for hubs, focus cities, fulfillment centers, and route costs.

The PuLP model used the CBC optimization solver. The solver returned the following results:

- Solver status: Optimal
- Minimum total cost: 182,376.25
- Decision variables: 192
- Constraints: 73
- Routes with positive shipments: 68

### B. Model Solution and Analysis

The solver returned an optimal solution with a total transportation cost of 182,376.25. The solution used 68 routes with positive shipment amounts.

#### B1. Constraint Validation

The shipment results were reviewed to confirm that the capacity, flow, and demand requirements were satisfied.

##### Hub Capacity

| Hub | Tons Shipped | Capacity | Remaining Capacity |
| --- | -----------: | -------: | -----------------: |
| CVG |       95,650 |   95,650 |                  0 |
| AFW |       38,097 |   44,350 |              6,253 |

CVG was used at full capacity, while AFW remained below its capacity.

##### Focus Cities

| Focus City | Inflow | Outflow | Capacity |
| ---------- | -----: | ------: | -------: |
| LEJ        | 24,470 |  24,470 |   85,000 |
| HYD        | 19,000 |  19,000 |   19,000 |
| SBD        |      0 |       0 |   36,000 |

Each focus city stayed within its capacity, and inflow equaled outflow.

The 65 fulfillment centers required a total of 133,747 tons. The model delivered 133,747 tons, so all demand requirements were met.

#### B2. Model Components

The model includes three groups of decision variables:

- `x[h,f]`: tons shipped from hub `h` to focus city `f`
- `y[h,c]`: tons shipped from hub `h` to fulfillment center `c`
- `z[f,c]`: tons shipped from focus city `f` to fulfillment center `c`

The objective function minimizes the total transportation cost.

The constraints make sure that:

- Hub capacity is not exceeded.
- Focus-city capacity is not exceeded.
- Focus-city inflow equals outflow.
- Each fulfillment center receives its required demand.
- Shipment amounts are not negative.

#### B3. Expected Results

The solution was close to what I expected. CVG was used at full capacity, HYD was also used at full capacity, and SBD was not used.

The model met all demand without exceeding any capacity limits. Some routes had the same cost, so another optimal solution could use different routes while producing the same total cost.


### C. Reflection

Developing the model mostly matched what I expected. The mathematical model from Task 2 translated into Python without major changes.

The part that took the most attention was handling the unavailable routes marked as N/A. Those routes had to be excluded so the solver would only use valid shipping options.

I also expected the model to return one clear shipping plan, but several routes had the same cost. Because of that, more than one solution could produce the same minimum total cost.

Overall, the process showed me that solving the model is only one part of the task. The results also need to be checked to make sure the capacities, flow balance, and fulfillment-center demands are satisfied.

### References

The model is based on the WGU Amazon transportation scenario (Western Governors University, n.d.).

Western Governors University. (n.d.). *Amazon Distribution*.

PuLP. (n.d.). *PuLP documentation*.


### Appendix A: Python Code for Task 3

In [9]:
import pandas as pd
from pulp import (LpProblem, LpMinimize, LpVariable, lpSum, LpStatus, value)


# reading data from the excel
file_path = "Task3.xlsx"

centers = pd.read_excel(file_path, sheet_name="Centers").set_index("center_id")
hubs = pd.read_excel(file_path, sheet_name="Hubs").set_index("hub_id")
focus = pd.read_excel(file_path, sheet_name="Focus Cities").set_index("focus_id")
cost = pd.read_excel(file_path, sheet_name="Cost", na_values=["N/A"]).set_index("destination_id")

# creating dictionaries for costs
hub_focus_cost = {
    (h, f): cost.at[f, h]
    for h in hubs.index
    for f in focus.index
    if pd.notna(cost.at[f, h])
}

hub_center_cost = {
    (h, c): cost.at[c, h]
    for h in hubs.index
    for c in centers.index
    if pd.notna(cost.at[c, h])
}

focus_center_cost = {
    (f, c): cost.at[c, f]
    for f in focus.index
    for c in centers.index
    if pd.notna(cost.at[c, f])
}

# Creating the model
model = LpProblem("Amazon_Transportation", LpMinimize)

x = LpVariable.dicts("hub_to_focus", hub_focus_cost, lowBound=0)
y = LpVariable.dicts("hub_to_center", hub_center_cost, lowBound=0)
z = LpVariable.dicts("focus_to_center", focus_center_cost, lowBound=0)

# Objective function
model += (
    lpSum(hub_focus_cost[h, f] * x[h, f] for h, f in hub_focus_cost)
    + lpSum(hub_center_cost[h, c] * y[h, c] for h, c in hub_center_cost)
    + lpSum(focus_center_cost[f, c] * z[f, c] for f, c in focus_center_cost)
)

for h in hubs.index:
    model += (
        lpSum(x[h, f] for f in focus.index if (h, f) in hub_focus_cost)
        + lpSum(y[h, c] for c in centers.index if (h, c) in hub_center_cost)
        <= hubs.at[h, "capacity"]
    )

for f in focus.index:
    inflow = lpSum(x[h, f] for h in hubs.index if (h, f) in hub_focus_cost)
    outflow = lpSum(z[f, c] for c in centers.index if (f, c) in focus_center_cost)

    model += inflow <= focus.at[f, "capacity"]
    model += outflow == inflow

for c in centers.index:
    model += (
        lpSum(y[h, c] for h in hubs.index if (h, c) in hub_center_cost)
        + lpSum(z[f, c] for f in focus.index if (f, c) in focus_center_cost)
        == centers.at[c, "demand"]
    )

status = model.solve()

print("Solver status:", LpStatus[status])
print("Minimum total cost:", value(model.objective))
print("Decision variables:", len(model.variables()))
print("Constraints:", len(model.constraints))
print("Routes with positive shipments:", positive_routes)

positive_routes = 0

for variable in model.variables():
    if variable.varValue is not None and variable.varValue > 0:
        positive_routes += 1
        print(variable.name, variable.varValue)




Welcome to the CBC MILP Solver 
Version: 2.10.3 
Build Date: Dec 15 2019 

command line - /home/rob/Documents/WGU Projects/operations-research-projects/.venv/lib/python3.12/site-packages/pulp/apis/../solverdir/cbc/linux/i64/cbc /tmp/c4de29de60834c90af4e71ce22144fa3-pulp.mps -timeMode elapsed -solve -printingOptions all -solution /tmp/c4de29de60834c90af4e71ce22144fa3-pulp.sol (default strategy 1)
At line 2 NAME          MODEL
At line 3 ROWS
At line 78 COLUMNS
At line 659 RHS
At line 733 BOUNDS
At line 734 ENDATA
Problem MODEL has 73 rows, 192 columns and 388 elements
Coin0008I MODEL read with 0 errors
Option for timeMode changed from cpu to elapsed
Presolve 59 (-14) rows, 173 (-19) columns and 348 (-40) elements
Perturbing problem by 0.001% of 1.6 - largest nonzero change 0.00010083972 ( 0.020167943%) - largest zero change 5.0652857e-05
0  Obj 108142.7 Primal inf 96868.101 (54) Dual inf 1.8999118 (2)
50  Obj 180858.51 Primal inf 1618.2 (13)
64  Obj 182382.08
Optimal - objective value 18